# EXP-001 - H1: LightGBM on Identical Numeric Features

**Hypothesis H1** (pre-registered in `docs/kaggle/research.md`): swapping sklearn GB for LightGBM
on the *same* numeric-only feature set, with native NaN handling replacing `fillna(0)`, yields
private LB delta-AUC >= +0.015 over EXP-000 (i.e. private >= 0.890). No new signal is added -
this isolates model class + imputation.

**Protocol** (`docs/kaggle/validation-protocol.md` v1): Scheme A holdout identical to EXP-000
(DeLong sample); Scheme B month-wise GroupKFold reported for the first time (feeds H4).
The holdout is NEVER used for model selection: early stopping runs on the last month *inside*
the train partition, then the model is refit on the full train partition at best_iter * 1.1.

**Anchors**: EXP-000 holdout AUC 0.8614; LB public/private 0.8896 / 0.8749 (SUB-001).

**Config registered before running**: lr 0.05, num_leaves 192, min_data_in_leaf 100,
feature_fraction 0.8, bagging_fraction 0.8, bagging_freq 1, seed 42, ES patience 200.

In [ ]:
import os
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SPLIT_QUANTILE = 0.8
SECONDS_PER_MONTH = 86400 * 30.44
EXCLUDE_COLS = {"isFraud", "TransactionID", "TransactionDT"}

LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=192,
    min_data_in_leaf=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    seed=42,
    n_jobs=-1,
    verbosity=-1,
)
MAX_ROUNDS = 5000
ES_PATIENCE = 200

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if ON_KAGGLE:
    hits = sorted(Path("/kaggle/input").rglob("train_transaction.csv"))
    if not hits:
        raise FileNotFoundError("Competition data not attached (Add Input -> Competitions).")
    DATA_DIR = hits[0].parent
else:
    DATA_DIR = Path("../../../data/raw")
print(f"Data dir: {DATA_DIR}")

## 1. Data, split and feature list

Identical to EXP-000 except the imputation: NaN is KEPT (LightGBM handles it natively).
Float32 downcast to halve memory; NaN survives the cast.

In [ ]:
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
df = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity

df["DT_M"] = (df["TransactionDT"] / SECONDS_PER_MONTH).astype(int)
cutoff = df["TransactionDT"].quantile(SPLIT_QUANTILE)
train_mask = (df["TransactionDT"] < cutoff).to_numpy()

feature_list = [
    c for c in df.columns if df[c].dtype != "O" and c not in EXCLUDE_COLS and c != "DT_M"
]
print(f"Features: {len(feature_list)} (expected 400, matching EXP-000 on this env)")

X = df[feature_list].astype("float32")  # NaN preserved - no fillna by design (H1)
y = df["isFraud"].astype(int).to_numpy()
months = df["DT_M"].to_numpy()
trans_ids = df["TransactionID"].to_numpy()
del df
print(f"Train: {train_mask.sum():,} | Holdout: {(~train_mask).sum():,} | Months: {sorted(set(months))}")

## 2. Scheme B - month-wise GroupKFold (feeds H4)

Leave-one-month-out over the full training set. Early stopping per fold on the held-out month,
as pinned in the protocol. Per-fold AUC and mean +/- std are the Scheme B record.

In [ ]:
fold_aucs, fold_best_iters = [], []
t0 = time.time()
for m in sorted(set(months)):
    tr = months != m
    va = ~tr
    clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
    clf.fit(
        X[tr], y[tr],
        eval_set=[(X[va], y[va])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
    )
    auc = roc_auc_score(y[va], clf.predict_proba(X[va])[:, 1])
    fold_aucs.append(auc)
    fold_best_iters.append(clf.best_iteration_)
    print(f"fold month={m}: AUC={auc:.4f}  best_iter={clf.best_iteration_}  ({(time.time()-t0)/60:.1f} min elapsed)")

scheme_b_mean, scheme_b_std = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
print(f"\nScheme B GroupKFold: {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")

## 3. Scheme A model - early stopping INSIDE the train partition

The last month of the train window is the ES validation set; the holdout never touches model
selection. The final Scheme A model is refit on the full train partition at best_iter * 1.1
(standard compensation for the extra data).

In [ ]:
train_months = months[train_mask]
es_month = train_months.max()
sub_tr = train_mask & (months < es_month)
es_va = train_mask & (months == es_month)

es_clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
es_clf.fit(
    X[sub_tr], y[sub_tr],
    eval_set=[(X[es_va], y[es_va])],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
)
best_iter = es_clf.best_iteration_
final_rounds = max(int(best_iter * 1.1), 100)
print(f"ES month: {es_month} | best_iter: {best_iter} | refit rounds: {final_rounds}")

model = lgb.LGBMClassifier(n_estimators=final_rounds, **LGB_PARAMS)
model.fit(X[train_mask], y[train_mask])

val_proba = model.predict_proba(X[~train_mask])[:, 1]
holdout_auc = roc_auc_score(y[~train_mask], val_proba)
print(f"Scheme A holdout ROC-AUC: {holdout_auc:.4f}  (EXP-000: 0.8614)")

## 4. DeLong test vs EXP-000

Inline mirror of `src/models/delong.py` (the kernel cannot import the repo package).
EXP-000 holdout predictions come from the attached kernel source, aligned by `TransactionID`.

In [ ]:
from scipy import stats

def _midrank(x):
    order = np.argsort(x, kind="mergesort")
    xs = x[order]
    n = len(x)
    rs = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and xs[j] == xs[i]:
            j += 1
        rs[i:j] = 0.5 * (i + j - 1) + 1.0
        i = j
    out = np.empty(n)
    out[order] = rs
    return out

def _components(y_true, scores):
    pos, neg = scores[y_true == 1], scores[y_true == 0]
    m, n = len(pos), len(neg)
    ra = _midrank(np.concatenate([pos, neg]))
    auc = (ra[:m].sum() - m * (m + 1) / 2.0) / (m * n)
    v10 = (ra[:m] - _midrank(pos)) / n
    v01 = 1.0 - (ra[m:] - _midrank(neg)) / m
    return auc, v10, v01

def delong_roc_test(y_true, sa, sb):
    auc_a, v10a, v01a = _components(y_true, sa)
    auc_b, v10b, v01b = _components(y_true, sb)
    m, n = len(v10a), len(v01a)
    var_a = np.var(v10a, ddof=1) / m + np.var(v01a, ddof=1) / n
    var_b = np.var(v10b, ddof=1) / m + np.var(v01b, ddof=1) / n
    cov = np.cov(v10a, v10b, ddof=1)[0, 1] / m + np.cov(v01a, v01b, ddof=1)[0, 1] / n
    delta = auc_a - auc_b
    se = np.sqrt(max(var_a + var_b - 2 * cov, 0.0))
    z = delta / se if se > 0 else 0.0
    p = 2 * stats.norm.sf(abs(z)) if se > 0 else 1.0
    half = stats.norm.ppf(0.975) * se
    return auc_a, auc_b, delta, (delta - half, delta + half), z, p

exp000_hits = sorted(Path("/kaggle/input").rglob("holdout_pred_exp000.csv")) if ON_KAGGLE else []
if not exp000_hits:
    raise FileNotFoundError("holdout_pred_exp000.csv not found - attach the EXP-000 kernel as input.")
prev = pd.read_csv(exp000_hits[0])

cur = pd.DataFrame({"TransactionID": trans_ids[~train_mask], "y_true": y[~train_mask], "score": val_proba})
merged = cur.merge(prev, on="TransactionID", suffixes=("_new", "_old"))
assert len(merged) == len(cur), "holdout rows do not align with EXP-000"
assert (merged["y_true_new"] == merged["y_true_old"]).all(), "labels differ - alignment bug"

auc_a, auc_b, delta, ci, z, p = delong_roc_test(
    merged["y_true_new"].to_numpy(),
    merged["score_new"].to_numpy(),
    merged["score_old"].to_numpy(),
)
print(f"DeLong: AUC new={auc_a:.4f} old={auc_b:.4f}")
print(f"delta={delta:+.4f}  95% CI=[{ci[0]:+.4f}, {ci[1]:+.4f}]  z={z:.2f}  p={p:.2e}")
print(f"Internal H1 direction: {'SUPPORTED' if p < 0.05 and delta > 0 else 'NOT significant / wrong direction'} (LB direction still required)")

## 5. Artifacts and submission

In [ ]:
cur.to_csv("holdout_pred_exp001.csv", index=False)
print("Saved holdout_pred_exp001.csv (DeLong artifact for EXP-002)")

test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")
test_identity.columns = [c.replace("id-", "id_") for c in test_identity.columns]
df_test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity

X_test = df_test.reindex(columns=feature_list).astype("float32")  # NaN kept
test_proba = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({"TransactionID": df_test["TransactionID"], "isFraud": test_proba})
submission.to_csv("submission.csv", index=False)
print(f"Rows: {len(submission):,} (expected 506,691)")

print("\n=== EXP-001 summary (copy to registry) ===")
print(f"Scheme A holdout AUC : {holdout_auc:.4f}")
print(f"Scheme B GroupKFold  : {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")
print(f"Per-fold AUCs        : {[round(a, 4) for a in fold_aucs]}")
print(f"DeLong vs EXP-000    : delta={delta:+.4f} CI=[{ci[0]:+.4f},{ci[1]:+.4f}] p={p:.2e}")

## 6. Before submitting (author checklist)

1. Copy the summary block into `docs/kaggle/experiment-registry.md` (EXP-001).
2. Complete SUB-002 in `docs/kaggle/submission-log.md` BEFORE upload.
3. Submit `submission.csv` (Output tab -> Submit to Competition, or CLI).
4. Fill public/private LB back into the log; H1 verdict = internal DeLong AND private LB
   delta >= +0.015 agreeing (private >= 0.890).